## Use BLEND to search for correlated variables

### Load libraries and define paths

In [1]:
import os
import sys
from pathlib import Path
import polars as pl
from tabulate import tabulate

In [2]:
sys.path.append(
    str(Path(os.path.abspath(os.path.curdir)).parent.absolute())
)

import blend
from blend.indexing import index_tables

In [3]:
data_path = Path(os.path.abspath(os.path.curdir)).parent.joinpath("examples", "example-data", "undata")

data_lake_path = data_path.joinpath("data-lake")
index_db_path = data_path.joinpath("undata.db")
logdir_path = data_path.joinpath("log")
queries_path = data_path.joinpath("queries")

data_path.exists()

True

In [4]:
from blend import BLEND
from blend.indexing import index_tables
from blend.utils import clean

### Instantiate the BLEND indexer

In [5]:
indexer = BLEND(index_db_path, clean_function_args={"lowercase": True, "filter_bad_tokens": True})

In [6]:
load_opts = {"ignore_errors": True}
index_tables(indexer, data_lake_path, True, None, 4, load_opts)

Parsing and storing tables: 100%|██████████| 53/53 [00:06<00:00,  7.87it/s]


(6.80141019821167, 1.3313579559326172, 8.132768154144287)

### Load the query dataset

We have some datasets in the _query_ folder:

In [7]:
queries = sorted(os.listdir(queries_path))

print('\n\n'.join(queries))

Adult literacy rate.csv

Government expenditure on education as % of GDP.csv

Mobile-cellular-telephone-subscriptions-per-100-inhabitants.csv


## Union Search

In [8]:
query_table_idx = 1
query_table_name = queries[query_table_idx]

qdf = pl.read_csv(os.path.join(queries_path, query_table_name))

print(f"Query dataset: {query_table_name}")

qdf

Query dataset: Government expenditure on education as % of GDP.csv


Country,Sub-region Name,Region Name,Year,Decade,Sex,Age group,Units of measurement,Value
str,str,str,i64,i64,str,str,str,f64
"""AFG""","""Southern Asia""","""Asia""",1982,1980,"""both""","""Not applicable""","""Percent""",1.72998
"""AFG""","""Southern Asia""","""Asia""",2010,2010,"""both""","""Not applicable""","""Percent""",4.51116
"""AFG""","""Southern Asia""","""Asia""",2011,2010,"""both""","""Not applicable""","""Percent""",4.08791
"""AFG""","""Southern Asia""","""Asia""",2012,2010,"""both""","""Not applicable""","""Percent""",3.12562
"""AFG""","""Southern Asia""","""Asia""",2013,2010,"""both""","""Not applicable""","""Percent""",4.54436
…,…,…,…,…,…,…,…,…
"""ZWE""","""Sub-Saharan Africa""","""Africa""",1990,1990,"""both""","""Not applicable""","""Percent""",12.45426
"""ZWE""","""Sub-Saharan Africa""","""Africa""",1992,1990,"""both""","""Not applicable""","""Percent""",22.32221
"""ZWE""","""Sub-Saharan Africa""","""Africa""",1994,1990,"""both""","""Not applicable""","""Percent""",44.33398


In [9]:
table = qdf.rows()
results = indexer.union_search(table, 10)

results_df = pl.DataFrame(results, orient='row')
print(f"Query dataset: {query_table_name}")
results_df

Query dataset: Government expenditure on education as % of GDP.csv


column_0
str
"""Youth unemployment, both sexes"""
"""Youth unemployment, women"""
"""Youth unemployment, men"""
"""AIDS-related deaths"""
"""Coverage of people receiving A…"
"""Armed forces personnel, total"""
"""Government expenditure on educ…"
"""Average-dietary-energy-require…"
"""Commercial banks and other len…"


## Join-Correlation Search 

Given a query dataset composed by two columns, _Kq_ and _Xq_, 
we need to identify datasets on which we can perform a join on key _Kc_ and that have a numerical column _Xc_ that is highly correlated with _Xq_.

Just looking for columns with an high join-overlap doesn't address our needs.

Define which dataset we want to use:

In [10]:
query_table_idx = 0
query_table_name = queries[query_table_idx]

qdf = pl.read_csv(os.path.join(queries_path, query_table_name))

print(f"Query dataset: {query_table_name}")

qdf

Query dataset: Adult literacy rate.csv


Country,Sub-region Name,Region Name,Sex,Age group,Year,Decade,Source,Unit,Value
str,str,str,str,str,i64,i64,str,str,f64
"""AFG""","""Southern Asia""","""Asia""","""female""","""15+ yr""",2000,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",12.6
"""AFG""","""Southern Asia""","""Asia""","""male""","""15+ yr""",2000,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",43.1
"""ALB""","""Southern Europe""","""Europe""","""female""","""15+ yr""",2001,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",98.3
"""ALB""","""Southern Europe""","""Europe""","""male""","""15+ yr""",2001,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",99.2
"""DZA""","""Northern Africa""","""Africa""","""female""","""15+ yr""",2002,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",60.1
…,…,…,…,…,…,…,…,…,…
"""ZMB""","""Sub-Saharan Africa""","""Africa""","""male""","""15+ yr""",1990,1990,"""UNESCO_UIS Database_Sep2007""","""Percent""",73.0
"""ZWE""","""Sub-Saharan Africa""","""Africa""","""female""","""15+ yr""",2004,2000,"""UNESCO_UIS Database_Sep2007""","""Percent""",86.2
"""ZWE""","""Sub-Saharan Africa""","""Africa""","""female""","""15+ yr""",1992,1990,"""UNESCO_UIS Database_Sep2007""","""Percent""",78.5


Define on which key and target columns we will perform the search

In [11]:
target_column_name = 'Value'

target_column_name

'Value'

In [12]:
# key_column_name = 'Country or Area'
key_column_name = 'Sub-region Name'
# key_column_name = 'Region Name'

key_column_name

'Sub-region Name'

### What's inside the key column?

When working with joins, correlations, ..., the granularity level choosen for the search affects the final results.

In our geographical datasets, fine-grained searches at the country level yield different results compared to coarser-grained searches at sub-regional or regional level.

In particular, regional granularity isn't really useful.

In [13]:
qdf.get_column(key_column_name).unique().sort()

Sub-region Name
str
"""ANT"""
"""Central Asia"""
"""Eastern Asia"""
"""Eastern Europe"""
"""Latin America and the Caribbea…"
…
"""South-eastern Asia"""
"""Southern Asia"""
"""Southern Europe"""


### Perform a Correlation Search (based on QCR schema)

Run a correlation search on the query dataset. 

First, we group it by the selected key column, which will be used to identify _joinable_ tables.

Then the retrieved tables will be ranked by an _estimate_ of the Pearson correlation, called **Quadrant Count Ratio** (QCR) correlation

In [14]:
# GROUP BY key_column + MEAN ON target_column
grouped_qdf = qdf.group_by(key_column_name).agg(pl.col(target_column_name).mean())

# rename, just because it will be useful in later steps
grouped_qdf = grouped_qdf.rename({target_column_name: 'Value_left'})

grouped_qdf.sort(key_column_name)

Sub-region Name,Value_left
str,f64
"""ANT""",95.575
"""Central Asia""",98.633333
"""Eastern Asia""",89.4125
"""Eastern Europe""",98.255
"""Latin America and the Caribbea…",88.373333
…,…
"""South-eastern Asia""",84.733333
"""Southern Asia""",63.152778
"""Southern Europe""",95.426471


In [15]:
# extract the key column values
keys = grouped_qdf.get_column(key_column_name).to_list()

# extract the target column values
targets = grouped_qdf.get_column('Value_left').to_list()

keys[:3], targets[:3]

(['Southern Asia', 'Polynesia', 'Eastern Asia'],
 [63.15277777777778, 98.51666666666667, 89.4125])

Run the correlation search task: see the relative SQL query used under the hood at **blend.Operators.Seekers.Correlation**

In [16]:
results = indexer.correlation_search(keys, targets, 20)

results_df = pl.DataFrame(results, schema=['dataset', 'join_col_idx', 'target_col_idx', 'QCR'], orient='row')

results_df

dataset,join_col_idx,target_col_idx,QCR
str,i64,i64,f64
"""Gender parity index for adult …",1,9,0.857143
"""Adult literacy rate""",1,9,0.818182
"""Coverage of pregnant women who…",1,8,1.0
"""Bribery incidence (% of firms …",1,4,0.5
"""Gender parity index for adult …",1,6,-0.428571
…,…,…,…
"""Armed forces personnel, total""",1,3,1.0
"""Gender parity index for gross …",1,4,0.666667
"""Gender parity index for net en…",1,8,0.5


### Compare results with actual Pearson

In [ ]:
from scipy import stats


def compare_with_pearson(results: list) -> list:
    results_with_pearson = []

    # basically, for each record load the relative dataset and compute 
    # the exact Pearson correlation (we have all necessary information, 
    # from the column indexes for key and target columns to the actual dataset)
    for table_id, join_col_idx, target_col_idx, qcr in results:
        r_df = pl.scan_csv(data_lake_path.joinpath(f"{table_id}.csv"))

        # aggregate each dataset on its identified key column 
        # and compute the mean of the group
        r_df = r_df.group_by(pl.nth(join_col_idx)).agg(pl.nth(target_col_idx).mean()).collect()
        
        # rename the column (just to make simpler next steps)
        target_col_name = r_df.columns[1]
        r_df = r_df.rename({r_df.columns[0]: key_column_name, r_df.columns[1]: 'Value_right'})
        
        # join our query grouped dataframe with the retrieved one
        # (which is, as well, grouped)
        join = grouped_qdf.join(
            r_df, 
            on=key_column_name
        )

        # extract the numerical columns used to compute the correlation
        value_left = join.get_column('Value_left')
        value_right = join.get_column('Value_right')

        # it could happen that after aggregation an array is
        # constant: in this case, Pearson correlation is not
        # defined and in the end we will have to discard these
        # NaN values...
        statistics = stats.pearsonr(value_left, value_right)

        pearson = statistics.correlation
        p_value = statistics.pvalue

        results_with_pearson.append(
            [
                table_id, join_col_idx, target_col_idx, target_col_name, qcr, pearson, p_value
            ]
        )

    return results_with_pearson

In [ ]:
results_with_pearson = compare_with_pearson(results)

results_with_pearson_df = pl.DataFrame(
    results_with_pearson, 
    schema=['dataset', 'join_col_idx', 'target_col_idx', 'target_col_name', 'QCR', 'pearson', 'p_value'], 
    orient='row'
    ).with_row_index('rank')

results_with_pearson_df

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Slider, CustomJS
from bokeh.transform import factor_cmap
from bokeh.layouts import row

# Display Bokeh plots in Jupyter
output_notebook()

In [ ]:
# Categorize p-values
def categorize_pval(p):
    if p < 0.05:
        return "< 0.05"
    elif p < 0.5:
        return "0.05-0.5"
    else:
        return ">=0.5"

df_pd = results_with_pearson_df.with_columns(
    pl.col('p_value').map_elements(categorize_pval, pl.String).alias('p_category')
).to_pandas()

# Convert to Bokeh ColumnDataSource
source = ColumnDataSource(df_pd)
source_all = ColumnDataSource(df_pd)

# Define color mapping for categories
categories = ["< 0.05", "0.05-0.5", ">=0.5"]
colors = ["red", "orange", "blue"]

# Create interactive plot
p = figure(
    title="Estimated vs Actual Pearson (interactive)",
    x_axis_label="Pearson",
    y_axis_label="Estimated Pearson",
    width=700,
    height=700,
    tools="pan,wheel_zoom,box_zoom,reset"
)

# Add scatter with hover tool
renderer = p.scatter(
    x="pearson",
    y="QCR",
    source=source,
    size=8,
    legend_field="p_category",
    fill_alpha=0.7,
    color=factor_cmap("p_category", palette=colors, factors=categories)
)

# Add identity line y=x
p.line([-1, 1], [-1, 1], line_dash="dashed", line_color="black")

# Add hover tool
hover = HoverTool(
    renderers=[renderer],
    tooltips=[
        ("Pearson", "@pearson{0.000}"),
        ("Estimate", "@QCR{0.000}"),
        ("p-value", "@p_value"),
        ("p-category", "@p_category"),
        ("target_name", "@target_col_name"),
        ("dataset", "@dataset"),
        ("rank", "@rank")
    ]
)
p.add_tools(hover)

p.legend.title = "p-value bins"
p.legend.location = "top_left"


# Slider widget for filtering by rank
slider = Slider(start=df_pd["rank"].min(),
                end=df_pd["rank"].max(),
                value=df_pd["rank"].max(),
                step=1,
                title="Max rank in top-K")

# Reassign the ENTIRE data dict
callback = CustomJS(args=dict(source=source, source_all=source_all, slider=slider), code="""
    const A = source_all.data;
    const max_pos = slider.value;

    const idx = [];
    const n = A['rank'].length;
    for (let i = 0; i < n; i++) {
        if (A['rank'][i] < max_pos) idx.push(i);
    }

    function pick(key) { return idx.map(i => A[key][i]); }

    source.data = {
        pearson: pick('pearson'),
        QCR: pick('QCR'),
        p_value: pick('p_value'),
        p_category: pick('p_category'),
        target_col_name: pick('target_col_name'),
        dataset: pick('dataset'),
        rank: pick('rank'),
    };
""")

slider.js_on_change("value", callback)

# Layout (plot + slider)
layout = row(p, slider)

print(f"Query dataset: {query_table_name}")
show(layout)

The QCR scheme gives an approximation of the Pearson correlation between two numerical variables. However, before going on with any analysis, the p-value should be checked to verify that the correlation could be considered statistically significant, and not spurious.

In general, this is a **pure statistical method**, and no information on the actual meaning of a variable is involved. Thus, a final check is always required before using the retrieved columns.